In [1]:
%%capture
!pip install lightgbm -q
!pip install optuna

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import f1_score
import optuna

In [3]:
X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')

y_train = pd.read_csv('/content/y_train.csv')
y_test = pd.read_csv('/content/y_test.csv')

In [4]:
X_train['text_cleaned'] = X_train['text'].str.replace(r'\r|\n', ' ', regex=True)
X_test['text_cleaned'] = X_test['text'].str.replace(r'\r|\n', ' ', regex=True)

In [7]:
X_train['text_cleaned']

,text_cleaned
0,Eve A True Story Told by the Beadle of the - C...
1,Proofreading Team. The Athenian Society ARISTO...
2,Table of Contents I. The Gun Club II. Presiden...
3,ORESTES. PYLADES. ARKAS. ACT THE FIRST Scene 1...
4,"Bishop of Worcester, to Lord Cromwell, on the ..."
...,...
387,Adventure of the Cardboard Box By Sir Arthur C...
388,"Ryan And Samson called unto the LORD, and said..."
389,Online Distributed Proofreading Team at http:/...
390,OF MID-LOTHIAN By Walter Scott TALES OF MY LAN...


## TF-IDF

In [24]:
vectorizer = TfidfVectorizer(min_df=0.05, max_df=0.9, stop_words='english')
tfidf_matr_train = vectorizer.fit_transform(X_train['text_cleaned'])
tfidf_matr_test = vectorizer.transform(X_test['text_cleaned'])
tfidf_matr_train.shape, tfidf_matr_test.shape

((392, 23894), (99, 23894))

In [10]:
y_train

,author
0,Nikolay_Gogol
1,Aristophanes
2,Jules_Verne
3,Johann_Wolfgang_Von_Goethe
4,Mark_Twain
...,...
387,Arthur_Conan_Doyle
388,Stephen_King
389,Walter_Scott
390,Walter_Scott


In [11]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train['author'])
y_test = label_encoder.transform(y_test['author'])

In [13]:
model_lgb_tfidf = LGBMClassifier(random_state=5)

model_lgb_tfidf.fit(tfidf_matr_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Выходные данные были обрезаны до нескольких последних строк (5000).
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

LGBMClassifier(random_state=5)

In [15]:
y_pred = model_lgb_tfidf.predict(tfidf_matr_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
accuracy, f1

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(0.7373737373737373, 0.6815205788467821)

## Теперь попробуем применить мешок слов: BoW

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
vectorizer = CountVectorizer(min_df=0.05, max_df=0.9, stop_words='english')
X_train_bow = vectorizer.fit_transform(X_train['text_cleaned'])
X_test_bow = vectorizer.transform(X_test['text_cleaned'])

In [8]:
X_train_bow = X_train_bow.astype('float32')
X_test_bow = X_test_bow.astype('float32')

In [9]:
model_lgb_bow = LGBMClassifier(random_state=42)
model_lgb_bow.fit(X_train_bow, y_train)

y_pred_bow = model_lgb_bow.predict(X_test_bow)

accuracy_bow = accuracy_score(y_test, y_pred_bow)
f1_bow = f1_score(y_test, y_pred_bow, average='weighted')

accuracy_bow, f1_bow

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Выходные данные были обрезаны до нескольких последних строк (5000).
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(0.7070707070707071, 0.6594021018263443)

На tf-idf качетсво было лучше

## sentence-BERT

In [10]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 855.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
X_train['embeddings'] = X_train['text_cleaned'].apply(lambda x: model.encode(x))
X_test['embeddings'] = X_test['text_cleaned'].apply(lambda x: model.encode(x))

In [13]:
X_train_bert = np.vstack(X_train['embeddings'].values)
X_test_bert = np.vstack(X_test['embeddings'].values)

In [14]:
model_lgb_bert = LGBMClassifier(random_state=5)

In [15]:
model_lgb_bert.fit(X_train_bert, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Выходные данные были обрезаны до нескольких последних строк (5000).
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

LGBMClassifier(random_state=5)

In [16]:
y_pred_bert = model_lgb_bert.predict(X_test_bert)

accuracy_bert = accuracy_score(y_test, y_pred_bert)
f1_bert = f1_score(y_test, y_pred_bert, average='weighted')

accuracy_bert, f1_bert

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(0.37373737373737376, 0.3352824309346048)

Берт показал самые худшие метрики

In [18]:
import re

X_train['text_cleaned'] = X_train['text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x.lower()))
X_test['text_cleaned'] = X_test['text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x.lower()))

## Попробую снизить размерность

In [21]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=300)
X_train_svd = svd.fit_transform(X_train_bow)
X_test_svd = svd.transform(X_test_bow)

In [22]:
model_lgb_svd = LGBMClassifier(random_state=5)
model_lgb_svd.fit(X_train_svd, y_train)

y_pred_svd = model_lgb_svd.predict(X_test_svd)

accuracy_svd = accuracy_score(y_test, y_pred_svd)
f1_svd = f1_score(y_test, y_pred_svd, average='weighted')

accuracy_svd, f1_svd

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Выходные данные были обрезаны до нескольких последних строк (5000).
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(0.5959595959595959, 0.5673677184372371)

In [25]:
svd = TruncatedSVD(n_components=300)
X_train_svd = svd.fit_transform(tfidf_matr_train)
X_test_svd = svd.transform(tfidf_matr_test)

In [27]:
model_lgb_svd = LGBMClassifier(random_state=5, verbose=-1)
model_lgb_svd.fit(X_train_svd, y_train)

y_pred_svd = model_lgb_svd.predict(X_test_svd)

accuracy_svd = accuracy_score(y_test, y_pred_svd)
f1_svd = f1_score(y_test, y_pred_svd, average='weighted')

accuracy_svd, f1_svd

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(0.5959595959595959, 0.528360016502309)

## Добавлю числовые признаки

In [28]:
X_train['text_length'] = X_train['text_cleaned'].apply(len)
X_test['text_length'] = X_test['text_cleaned'].apply(len)

In [29]:
X_train['punctuation_count'] = X_train['text'].apply(lambda x: len(re.findall(r'[!?]', x)))
X_test['punctuation_count'] = X_test['text'].apply(lambda x: len(re.findall(r'[!?]', x)))

In [30]:
from scipy.sparse import hstack

vectorizer = TfidfVectorizer(min_df=0.05, max_df=0.9, stop_words='english')
tfidf_matr_train = vectorizer.fit_transform(X_train['text_cleaned'])
tfidf_matr_test = vectorizer.transform(X_test['text_cleaned'])

numeric_features_train = np.array(X_train[['text_length', 'punctuation_count']])
numeric_features_test = np.array(X_test[['text_length', 'punctuation_count']])

X_train_combined = hstack([tfidf_matr_train, numeric_features_train])
X_test_combined = hstack([tfidf_matr_test, numeric_features_test])


In [31]:
model_lgb_num = LGBMClassifier(random_state=5, verbose=-1)
model_lgb_num.fit(X_train_combined, y_train)

y_pred_num = model_lgb_num.predict(X_test_combined)

accuracy_num = accuracy_score(y_test, y_pred_num)
f1_num = f1_score(y_test, y_pred_num, average='weighted')

accuracy_num, f1_num

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/lightgbm/bas

(0.7474747474747475, 0.6907135183427519)

Лучшие метрики получились с использованием tf-idf и числовыми признаками, сама метрика не улучшилась +- осталась такой же.